# 01 - Ingesta CNMC -> raw

Este notebook descarga los recursos publicos de CNMC relacionados con MIBGAS. La ventana historica comun se carga desde `config/modelo.yml`. Los JSON raw conservan la respuesta y la hora de captura.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / 'requirements.txt').is_file() and (candidate / 'notebooks').is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('No se encontro la raiz del proyecto.')

sys.path.insert(0, str(PROJECT_ROOT))
from src.project import find_project_root

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
from src.config.cnmc import load_cnmc_config
from src.ingesta.cnmc import CnmcIngestionConfig, CnmcResourceConfig, ingest_resource

CNMC_CONFIG_PATH = PROJECT_ROOT / "config" / "cnmc.yml"
MODEL_CONFIG_PATH = PROJECT_ROOT / "config" / "modelo.yml"
CONFIG = load_cnmc_config(CNMC_CONFIG_PATH, MODEL_CONFIG_PATH)
CNMC = CONFIG["cnmc"]
MODEL = CNMC["model"]
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cnmc"
print(f"Ventana historica: {MODEL['history_start']} a {MODEL['history_end']}")
print(f"Destino raw: {RAW_DIR}")

Ventana historica: 2024-10-01 a 2026-09-30
Destino raw: c:\Users\zhen.xiang\OneDrive - Accenture\Documents\SAT\Energy\odb-challenge-energy-optimizer\data\raw\cnmc


In [ ]:
INGESTION = CNMC['ingestion']
ingestion_config = CnmcIngestionConfig(
    raw_dir=RAW_DIR,
    base_url=INGESTION['base_url'],
    page_size=INGESTION['page_size'],
    retries=INGESTION['retries'],
    timeout=INGESTION['timeout_seconds'],
    pause_seconds=INGESTION['pause_seconds'],
    force=INGESTION['force'],
)

results = []
for name, settings in CNMC['resources'].items():
    resource = CnmcResourceConfig(name=name, **settings)
    results.append(ingest_resource(resource, ingestion_config))

results

In [ ]:
for status, path, count in results:
    print(f'{status}: {path} ({count:,} registros)')

## 2. Resumen visual de la ingesta

La tabla y el gráfico permiten comprobar rápidamente qué recursos se han descargado y cuántos registros contiene cada respuesta.

In [ ]:
ingestion_frame = pd.DataFrame(results, columns=["status", "path", "records"])
ingestion_frame

try:
    import matplotlib.pyplot as plt

    ingestion_frame.plot.bar(
        x="path",
        y="records",
        legend=False,
        figsize=(10, 4),
        color="#457b9d",
        title="Registros descargados por recurso CNMC",
    )
    plt.ylabel("Registros")
    plt.xlabel("")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib no está instalado; se conserva la tabla de resultados.")